# Code-Along: Snowpark Basics

Follow along with the instructor. Type each cell yourself, run it, and watch what comes back. Keep this notebook afterwards as your Snowpark reference.

**What Snowpark is:** a DataFrame API that runs inside Snowflake. You write DataFrame code in Python; Snowpark translates it to SQL and Snowflake's warehouses execute it. Your laptop never downloads the data until you explicitly ask.

**Why you care:** previously you used the Python connector and `write_pandas`, where pandas did the work on your machine and Snowflake stored the result. Snowpark flips that: the work happens where the data lives. This is the same idea you will meet again in Databricks with PySpark, and the API will look almost identical.

**Before you start:** this notebook expects the `W5D1_STOCK` table in your Snowflake schema (or run the setup SQL in Step 0 below). Your `snow.cfg` from Week 4 should sit next to this notebook (copy it, never commit it; the root `.gitignore` already ignores it).

## 0. Optional Table Setup (run once in Snowsight if table is missing)

```sql
USE ROLE DE;
USE WAREHOUSE COMPUTE_WH;
USE DATABASE TECHCATALYST;
USE SCHEMA TECHCATALYST.<YOUR_NAME>;

CREATE OR REPLACE TRANSIENT TABLE W5D1_STOCK (
  trade_date  DATE,
  ticker      VARCHAR(10),
  close_price NUMBER(10, 2)
);

INSERT INTO W5D1_STOCK VALUES
  ('2026-07-01', 'AAPL', 100.00),
  ('2026-07-02', 'AAPL', 102.50),
  ('2026-07-03', 'AAPL', 106.00),
  ('2026-07-06', 'AAPL', 108.20),
  ('2026-07-07', 'AAPL', 104.50);
```

## 1. Install and import

`snowflake-snowpark-python` is already in the root project. If your import fails, run the `uv add` line from the repository root in a terminal, not from this notebook.

In [ ]:
# If needed, from the repository root terminal:
# uv add snowflake-snowpark-python

from snowflake.snowpark import Session
from configparser import ConfigParser

## 2. Connect: a Session instead of a connection

Same `snow.cfg`, same parameters as Friday. The difference is what you get back: the connector gave you a `connection` for sending SQL strings; Snowpark gives you a `Session`, the entry point for DataFrames.

In [ ]:
config = ConfigParser()
config.read("snow.cfg")
params = dict(config["DEV"])

session = Session.builder.configs(params).create()
session

In [ ]:
print(session.get_current_role())
print(session.get_current_warehouse())
print(session.get_current_database())
print(session.get_current_schema())

## 3. Point at a table

`session.table` does not run a query. It creates a Snowpark DataFrame: a description of data you might ask for. Note the type: this is not pandas.

In [ ]:
stock = session.table("W5D1_STOCK")
type(stock)

In [ ]:
# .show() is an action: NOW a query runs in Snowflake and prints a small preview
stock.show(5)

## 4. Lazy evaluation: the big idea

Transformations (`select`, `filter`, `sort`) build a plan. Nothing executes until an action (`show`, `count`, `collect`, `to_pandas`) asks for results. This is exactly how Spark works too.

In [ ]:
from snowflake.snowpark.functions import col

# Three transformations, zero queries executed so far
gains = (
    stock
    .filter(col("CLOSE_PRICE") > 105)
    .select("TRADE_DATE", "CLOSE_PRICE")
    .sort(col("CLOSE_PRICE").desc())
)
type(gains)

In [ ]:
# Peek at the SQL Snowpark wrote for us
gains.explain()

In [ ]:
# The action: this is the moment Snowflake runs the query
gains.show()

## 5. The same thinking, third notation

This morning you wrote `GROUP BY` in SQL and `groupby` in pandas. Here is the Snowpark spelling. One idea, three notations.

In [ ]:
from snowflake.snowpark.functions import avg, max as max_, min as min_

stock.agg(
    min_("CLOSE_PRICE").alias("LOWEST_CLOSE"),
    max_("CLOSE_PRICE").alias("HIGHEST_CLOSE"),
    avg("CLOSE_PRICE").alias("AVG_CLOSE"),
).show()

Window functions exist here too. This is the `LAG` drill from Activity 2, in Snowpark:

In [ ]:
from snowflake.snowpark import Window
from snowflake.snowpark.functions import lag

w = Window.order_by("TRADE_DATE")

daily = stock.with_column("PREV_CLOSE", lag("CLOSE_PRICE").over(w)) \
             .with_column("DAILY_CHANGE", col("CLOSE_PRICE") - col("PREV_CLOSE"))
daily.show()

## 6. Raw SQL is still right there

When SQL is the clearest way to say something, say it in SQL. `session.sql` returns the same kind of DataFrame.

In [ ]:
session.sql("SELECT COUNT(*) AS N FROM W5D1_STOCK").show()

## 7. Why this matters: pushdown on big data

`SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS` has 1.5 million rows. Watch what does NOT happen: no download, no memory spike. The count and the aggregation run in the warehouse; only the small result travels to us.

In [ ]:
orders = session.table("SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS")
orders.count()

In [ ]:
from snowflake.snowpark.functions import year, sum as sum_

(
    orders
    .group_by(year("O_ORDERDATE").alias("ORDER_YEAR"))
    .agg(sum_("O_TOTALPRICE").alias("TOTAL_SALES"))
    .sort("ORDER_YEAR")
    .show()
)

Compare that with Friday's pattern: with the connector plus pandas, those 1.5 million rows would have crossed the network into your laptop's RAM before you could aggregate them.

## 8. The boundary: to_pandas

`to_pandas` pulls results into your laptop's memory. The rule: aggregate first in Snowflake, then convert the small result when you want pandas tools (plotting, scikit-learn, a quick export).

In [ ]:
yearly_pd = (
    orders
    .group_by(year("O_ORDERDATE").alias("ORDER_YEAR"))
    .agg(sum_("O_TOTALPRICE").alias("TOTAL_SALES"))
    .sort("ORDER_YEAR")
    .to_pandas()
)
print(type(yearly_pd))
yearly_pd

## 9. Writing back

Same modes you saw with `write_pandas` on Friday: `overwrite`, `append`, `errorifexists`, `ignore`, `truncate`.

In [ ]:
daily.write.mode("overwrite").save_as_table("W5D1_STOCK_DAILY")
session.table("W5D1_STOCK_DAILY").count()

## 10. Choosing your tool

| You want to | Reach for |
|---|---|
| Run SQL strings from Python, fetch small results | Python connector |
| Push a pandas DataFrame you built locally into a table | `write_pandas` |
| Let pandas `read_sql` / `to_sql` work against Snowflake | SQLAlchemy engine |
| Transform data that lives in Snowflake without downloading it | Snowpark |

One sentence to remember: **Snowpark moves the code to the data; pandas moves the data to the code.**

In [ ]:
session.close()